# 01 — Prepare and inspect the historical lake data
This is the first research-data notebook. It will read the original files without editing them and create clearly labelled **provisional** outputs.

**Question:** Can we assemble a reliable historical inventory table before comparing models?

The target is membership of the historical ICIMOD PDGL list: **1 = listed; 0 = not listed**. Zero does not mean proven safe. Our initial predictors will be area, elevation and lake type. IDs, coordinates and basin remain available for checks and mapping, but are not initial predictors.

The current Nepal selection uses the source Country field. A boundary check remains necessary before finalising the study population. No models are trained here.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/model_analysis.py').is_file())
DATA = ROOT.parent / 'outputs' / 'GLOF_MSc_datasets'
REPORTS = ROOT / 'reports'
PROCESSED = ROOT / 'data' / 'processed'
REPORTS.mkdir(exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'verification' / 'matplotlib'))
print('Project:', ROOT)


In [ ]:
import hashlib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import Geod

def unique_file(folder, pattern):
    paths = sorted((DATA / folder).rglob(pattern))
    assert len(paths) == 1, paths
    return paths[0]
base_path = unique_file('00_ICIMOD_base_2015','*.shp')
pdgl_path = unique_file('01_ICIMOD_PDGL','*.shp')
base = gpd.read_file(base_path)
pdgl = gpd.read_file(pdgl_path)
for frame in [base, pdgl]:
    assert frame.crs is not None
    assert frame.GL_ID.notna().all() and frame.GL_ID.is_unique
    assert frame.GL_ID.astype(str).str.strip().eq(frame.GL_ID).all()
assert set(pdgl.GL_ID).issubset(set(base.GL_ID)), 'Unmatched PDGL IDs need investigation.'
print('Inventory:', len(base), '| PDGL list:', len(pdgl))
print('CRS:', base.crs)
display(base.drop(columns='geometry').head())

## 1. Check the join
Use the unique GL_ID, not row position or the non-unique Id field. Compare shared attributes to catch conflicting matches. Predictors always come from the complete inventory, never from fields available only for listed lakes.

In [ ]:
shared = ['Country','Basin','Type','Area','Elevation','Latitude','Longitude']
joined = pdgl[['GL_ID']+shared].merge(base[['GL_ID']+shared], on='GL_ID', validate='one_to_one', suffixes=('_pdgl','_base'))
differences = []
for field in shared:
    left, right = joined[field+'_pdgl'], joined[field+'_base']
    equal = np.isclose(left, right, equal_nan=True) if pd.api.types.is_numeric_dtype(left) else left.eq(right)
    for _, row in joined.loc[~equal].iterrows():
        differences.append({'GL_ID':row.GL_ID,'field':field,'pdgl_value':row[field+'_pdgl'],'base_value':row[field+'_base']})
pd.DataFrame(differences, columns=['GL_ID','field','pdgl_value','base_value']).to_csv(REPORTS/'join_differences.csv',index=False)
print('Shared-attribute differences:', len(differences))
base['historical_pdgl'] = base.GL_ID.isin(pdgl.GL_ID).astype('int8')
lakes = base.loc[base.Country.astype(str).str.strip().eq('Nepal')].copy()
assert set(lakes.Basin) <= {'Koshi','Gandaki','Karnali'}
print('Provisional Nepal lakes:',len(lakes),'| listed:',int(lakes.historical_pdgl.sum()))

## 2. Check missing values, geometry and area
A valid polygon is not proof of a correct geographic boundary. The area comparison uses WGS84 ellipsoidal polygon areas, never square degrees. Ratios close to one support interpreting the inventory Area as km², but do not replace source documentation. Invalid geometry is flagged, not silently repaired.

In [ ]:
required = ['Area','Elevation','Type','Latitude','Longitude']
missing = lakes[required].isna().sum().to_frame('missing_count')
display(missing)
missing.to_csv(REPORTS/'missing_values.csv')
valid = lakes.geometry.notna() & ~lakes.geometry.is_empty & lakes.geometry.is_valid
lakes['geometry_valid'] = valid
geod = Geod(ellps='WGS84')
geo = lakes.to_crs(4326)
lakes['computed_area_km2'] = [abs(geod.geometry_area_perimeter(g)[0])/1e6 if ok else np.nan for g,ok in zip(geo.geometry,valid)]
lakes['area_ratio_source_to_computed'] = lakes.Area / lakes.computed_area_km2
lakes['area_check_flag'] = ~lakes.area_ratio_source_to_computed.between(0.9,1.1)
lakes['numeric_check_flag'] = (~np.isfinite(lakes[['Area','Elevation','Latitude','Longitude']]).all(axis=1) | (lakes.Area<=0) | (lakes.Elevation<=0) | ~lakes.Latitude.between(-90,90) | ~lakes.Longitude.between(-180,180))
print('Invalid geometries:',int((~valid).sum()))
print('Area ratio outside 0.9–1.1:',int(lakes.area_check_flag.sum()))
print('Numeric flags:',int(lakes.numeric_check_flag.sum()))
display(lakes.area_ratio_source_to_computed.describe())
flags = lakes.loc[~valid | lakes.area_check_flag | lakes.numeric_check_flag].drop(columns='geometry')
flags.to_csv(REPORTS/'lake_quality_flags.csv',index=False)

## 3. Describe the population before modelling
These are descriptive counts, not model performance. Rare listed lakes make accuracy alone misleading. Type may partly reflect historical designation criteria, so the later comparison without type is important.

In [ ]:
by_basin = pd.crosstab(lakes.Basin,lakes.historical_pdgl).reindex(columns=[0,1],fill_value=0).rename(columns={0:'not_listed',1:'listed'})
by_type = pd.crosstab(lakes.Type,lakes.historical_pdgl).reindex(columns=[0,1],fill_value=0).rename(columns={0:'not_listed',1:'listed'})
display(by_basin); display(by_type)
by_basin.to_csv(REPORTS/'counts_by_basin.csv'); by_type.to_csv(REPORTS/'counts_by_type.csv')
fig, axes = plt.subplots(1,2,figsize=(11,4))
by_basin.plot.bar(ax=axes[0],color=['#91a8bd','#be4b37'])
axes[0].set_title('Historical inventory membership'); axes[0].set_ylabel('Lake count'); axes[0].tick_params(axis='x',rotation=0)
for label,color,name in [(0,'#91a8bd','Not listed'),(1,'#be4b37','Listed PDGL')]:
    subset=lakes[lakes.historical_pdgl.eq(label)]
    axes[1].scatter(subset.Longitude,subset.Latitude,s=7 if label==0 else 30,c=color,label=name,alpha=.65)
axes[1].set(xlabel='Longitude (°E)',ylabel='Latitude (°N)',title='Provisional inventory locations')
axes[1].legend(); fig.tight_layout()
fig.savefig(REPORTS/'figures'/'initial_inventory_overview.png',dpi=180)
plt.show()

## 4. Save provisional outputs and a reproducibility record
The CSV retains audit columns; the explicit initial feature list is only Area, Elevation and Type. Do not send the whole CSV directly to a model. The GeoPackage preserves geometry for the next boundary review. Originals remain unchanged.

In [ ]:
lakes['selection_status'] = 'provisional_country_attribute_boundary_pending'
lakes.drop(columns='geometry').to_csv(PROCESSED/'lakes_nepal_PROVISIONAL.csv',index=False)
gpkg = PROCESSED/'lakes_nepal_PROVISIONAL.gpkg'
lakes.to_file(gpkg,layer='lakes',driver='GPKG',mode='w')
manifest=[]
for path in [base_path,pdgl_path]:
    for f in sorted(path.parent.glob(path.stem+'.*')):
        manifest.append({'path':str(f.relative_to(ROOT.parent)),'bytes':f.stat().st_size,'sha256':hashlib.sha256(f.read_bytes()).hexdigest()})
summary={'inventory_records':len(base),'pdgl_records':len(pdgl),'provisional_nepal_records':len(lakes),'listed':int(lakes.historical_pdgl.sum()),'not_listed':int((lakes.historical_pdgl==0).sum()),'invalid_geometries':int((~valid).sum()),'area_flags':int(lakes.area_check_flag.sum()),'numeric_flags':int(lakes.numeric_check_flag.sum()),'join_attribute_differences':len(differences),'core_features':['Area','Elevation','Type'],'target':'historical_pdgl','ready_for_final_modelling':False,'remaining_checks':['Nepal boundary source and spatial membership, including border lakes','Resolve quality flags and confirm source units / observation and designation dates','Freeze final cohort and validation design before modelling'],'sources':manifest}
(REPORTS/'data_preparation_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps({k:v for k,v in summary.items() if k!='sources'},indent=2))

## What I should understand before continuing
1. What is the difference between an unlisted lake and a safe lake?
2. Why do we join by GL_ID rather than by row number?
3. Why are Rank, identifiers and positive-list-only fields excluded from predictors?
4. Why is this table still provisional?

**Next notebook:** verify the Nepal boundary and resolve flags, then freeze the final modelling table. Climate and recent lake changes remain separate later steps. No model results have been produced by this notebook.